# Punch out

Probe the resonator at low amplitude, then high amplitude. See if there's difference.

# General Input

In [ ]:
# LabOne Q:
from laboneq.simple import *

# Helpers:
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

import matplotlib.pyplot as plt
import numpy as np
import laboneq
print(laboneq.__version__)
import numpy as np
import matplotlib.pyplot as plt
import time
# Use this if you need to reset
# session.devices[f"SHFQC_{serial_num}"].factory_reset()

# Helper Functions

In [ ]:
# def create_readout_pulse(
#     qubit, duration = , amplitude=0.9
# ):
#     readout_pulse = pulse_library.const(
#         uid=f"readout_pulse_{qubit}",
#         length=length,
#         amplitude=amplitude,
#     )
#     return readout_pulse

# Descriptor and Basic Setup

In [ ]:
"""Descriptor for a QCCS consisting of a single SHFQC
"""
descriptor_shfqc = """ 
instruments:
  SHFQC:
  - address: DEV12296
    uid: device_shfqc

connections:
  device_shfqc:
    - iq_signal: q0/drive_line
      ports: SGCHANNELS/0/OUTPUT
    - iq_signal: q0/measure_line
      ports: [QACHANNELS/0/OUTPUT]
    - acquire_signal: q0/acquire_line
      ports: [QACHANNELS/0/INPUT]
"""
# Define and Load our Device Setup
device_setup = DeviceSetup.from_descriptor(
    descriptor_shfqc,
    server_host="127.0.0.1",  # ip address of the LabOne dataserver used to communicate with the instruments
    server_port="8004",  # port number of the dataserver - default is 8004
    setup_name="UCLA_SHFQC",  # setup name
)

# Are we emulating? or actually creating pulses? ****
emulate = True
# create and connect to session
session = Session(device_setup=device_setup)
session.connect(do_emulation=emulate)

# Experiment Definition

In [ ]:


def punch_out(
    exp_id             = "punch_out",
    average_exponent   = 5,
    acq_freq_sweep     = LinearSweepParameter(uid="acquisition_frequency_sweep_default", start=-600e6, stop=600e6, count=2001),
    acq_amp_sweep      = LinearSweepParameter(uid="acquisition_amplitude_sweep_default", start=0.1, stop=0.7, count=10),
    integration_length = 1e-6,
    ):

    readout_pulse = pulse_library.const(uid="readout_pulse",length=integration_length)
    # Create Experiment
    exp = Experiment(
        uid = exp_id,
        signals = [
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )
    with exp.sweep(uid='acq_amp_sweep', parameter=acq_amp_sweep):
        with exp.acquire_loop_rt(
            uid="freq_shots",
            count=pow(2, average_exponent),
            acquisition_type=AcquisitionType.SPECTROSCOPY,
            ):
            with exp.sweep(uid="acq_freq_sweep", parameter=acq_freq_sweep):
                with exp.section(uid="spectroscopy"):
                    exp.play(signal="measure", pulse=readout_pulse),
                    exp.acquire(signal="acquire", handle="res_spec"),
    return exp

In [ ]:
acq_lo_freq = 7.0e9
acq_res_fre = 7.10e9

acq_fre_swp_st  = (acq_res_fre - 100e6) - 7e9
acq_fre_swp_end = (acq_res_fre + 100e6) - 7e9

acq_fre_swp_cnt = 501


acq_freq_sweep   = LinearSweepParameter(
    uid="acquisition_frequency_sweep", 
    start=acq_fre_swp_st, 
    stop=acq_fre_swp_end, 
    count=acq_fre_swp_cnt
    )

acq_amp_sweep = LinearSweepParameter(
    uid="acquisition_amplitude_sweep", 
    start=0.1, 
    stop=0.7, 
    count=10
    )

one_punch_man = punch_out(
    average_exponent = 10,
    acq_freq_sweep = acq_freq_sweep,
    integration_length = 5e-6,
    )
print(acq_freq_sweep)

决定这个实验的名字叫一拳超人，嗯

所以punch out也可以是one_punch_man，opm

In [ ]:
map_q0 = {}
map_q0['measure'] = device_setup.logical_signal_groups["q0"].logical_signals["measure_line"]
map_q0['acquire'] = device_setup.logical_signal_groups["q0"].logical_signals["acquire_line"]
one_punch_man.set_signal_map(map_q0)

In [ ]:
exp_calibration = Calibration()

exp_calibration["measure"] = SignalCalibration(
    oscillator = Oscillator(uid = "qa_osc_0", 
                            frequency = acq_freq_sweep,
                            modulation_type=ModulationType.HARDWARE),
    local_oscillator = Oscillator(uid="qa_lo", 
                                  frequency = acq_lo_freq),
    range = -30,
    amplitude= acq_amp_sweep,
)

exp_calibration["acquire"] = SignalCalibration(
    range = -40,
    amplitude= 1.0
)

one_punch_man.set_calibration(exp_calibration)

In [ ]:
compiled_opm = session.compile(one_punch_man)

In [ ]:
plot_simulation(compiled_opm, 0, 20e-6)

In [ ]:
run_opm = session.run(compiled_opm)

In [ ]:
data_opm = run_opm.get_data("res_spec")
freq_opm = run_opm.get_axis("res_spec")[0]

fig, [ax1,ax2] = plt.subplots(2,1)
ax1.plot(freq_opm/1e6, np.real(data_opm), ".-r")
ax1.plot(freq_opm/1e6, np.imag(data_opm), ".-b")
ax2.plot(freq_opm/1e6, np.unwrap(np.angle(data_opm))/np.pi, "orange")
ax1.set_ylabel("Amplitude")
ax2.set_ylabel("Phase (pi rad)")
ax2.set_xlabel("frequency (MHz)")
ax1.grid()
ax2.grid()
plt.show()

# use .py file for a change

In [26]:
import sys
import os

import importlib.util

module_name = "BasicCharacterization"
module_file = "BasicCharacterization.py"

spec = importlib.util.spec_from_file_location(module_name, module_file)
if spec is not None:
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    try:
        spec.loader.exec_module(module)
        # Import all classes from the module
        for attr_name in dir(module):
            attr = getattr(module, attr_name)
            if isinstance(attr, type):
                globals()[attr_name] = attr
    except Exception as e:
        print(f"Failed to import {module_file}: {e}")
else:
    print(f"Could not find module spec for {module_file}")

In [43]:
punchout = PunchOut()

2.57.0


ScannerError: mapping values are not allowed in this context
  in "<unicode string>", line 5, column 16